In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,2:-1], df.iloc[:,1], test_size=0.2)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
X_train_tensor = torch.from_numpy(X_train_scaled)
X_test_tensor = torch.from_numpy(X_test_scaled)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [ ]:
class MySimpleNN():
  def __init__(self,X):
    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad = True)
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)
  def forward(self,X):
    z = torch.matmul(X,self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred
  def loss_function(X, y_pred, y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
    loss = -(y*torch.log(y_pred) + (1-y)*torch.log(1-y_pred)).mean()
    return loss

In [ ]:
learning_rate = 0.1
epochs = 25

In [ ]:
model = MySimpleNN(X_train_tensor)
for epoch in range(epochs):
  y_pred = model.forward(X_train_tensor)
  loss = model.loss_function(y_pred, y_train_tensor)
  loss.backward()
  with torch.no_grad():
    model.weights -= learning_rate*model.weights.grad
    model.bias -= learning_rate*model.bias.grad
  model.weights.grad.zero_()
  model.bias.grad.zero_()
  print(f'epoch: {epoch+1}, loss: {loss.item()}')

epoch: 1, loss: 3.8254536240523485
epoch: 2, loss: 3.703557020417541
epoch: 3, loss: 3.579747613114861
epoch: 4, loss: 3.447175859155599
epoch: 5, loss: 3.3123302698852486
epoch: 6, loss: 3.1766801577454506
epoch: 7, loss: 3.033673895286537
epoch: 8, loss: 2.8886580848766403
epoch: 9, loss: 2.7326558812228967
epoch: 10, loss: 2.5715420169732957
epoch: 11, loss: 2.4101438678015756
epoch: 12, loss: 2.2444881258098834
epoch: 13, loss: 2.076753150120725
epoch: 14, loss: 1.9123955610603436
epoch: 15, loss: 1.7507570953020086
epoch: 16, loss: 1.5990950843122038
epoch: 17, loss: 1.4590531699863114
epoch: 18, loss: 1.330714604432736
epoch: 19, loss: 1.2146479510717625
epoch: 20, loss: 1.1151481831791723
epoch: 21, loss: 1.032824252973172
epoch: 22, loss: 0.9672749441173971
epoch: 23, loss: 0.916959794028027
epoch: 24, loss: 0.879426957540986
epoch: 25, loss: 0.8518125905108099


In [ ]:
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'accuracy: {accuracy.item()}')

accuracy: 0.504001259803772
